Lab 03 — Escrow & payment channels (Module 3).

Receipt-gated debit modelled on AIMarketEscrow.
Run:  python labs/lab03_escrow_channel.py
Exercise: python labs/run_exercises.py --module m3

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alexar76/aimarket-courses/blob/main/smart-contracts-course/notebooks/{})


In [ ]:
# Setup — run this cell once per session
# Live oracle sandbox (clones alexar76/oracles) — run once per Colab session
import os
import subprocess
import sys

REPO = "https://github.com/alexar76/aimarket-courses.git"
DEST = "/content/aimarket-courses"

if not os.path.isdir(DEST):
    subprocess.run(["git", "clone", "-q", "--depth", "1", REPO, DEST], check=True)
WORKDIR = os.path.join(DEST, "smart-contracts-course")
os.chdir(WORKDIR)

# Live oracle sandbox — clone the AIMarket oracles this course calls, then install them.
if not os.path.isdir("_deps/oracles"):
    subprocess.run(["git", "clone", "-q", "--depth", "1",
                    "https://github.com/alexar76/oracles.git", "_deps/oracles"], check=True)
for _pkg in ["core", "oracles/chronos", "oracles/sortes", "oracles/platon/backend"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"_deps/oracles/{_pkg}"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[oracles,dev]"], check=True)
os.environ.setdefault("COURSE_LANG", "en")  # change to ru or es


In [ ]:
"""Lab 03 — Escrow & payment channels (Module 3).

Receipt-gated debit modelled on AIMarketEscrow.
Run:  python labs/lab03_escrow_channel.py
Exercise: python labs/run_exercises.py --module m3
"""

from __future__ import annotations



from courselib.contracts import EscrowChannel
from courselib.i18n import get_translator
from courselib.trace import Trace


def main() -> None:
    t = get_translator()
    trace = Trace()
    print(f"== {t('modules.m3.title')} ==")
    print(t("modules.m3.concept"))

    ch = EscrowChannel("ch-lab03", t("labs.lab03.depositor"))
    ch.open(1000)
    trace.log("open", balance=ch.balance)

    ch.debit(250, "rcpt-001", "hub-course", signature_valid=True)
    trace.log("debit", amount=250, balance=ch.balance, nonce=ch.nonce)

    try:
        ch.debit(250, "rcpt-001", "hub-course", signature_valid=True)
        replay_ok = True
    except ValueError as exc:
        replay_ok = False
        trace.log("replay_blocked", error=str(exc))
    print(f"{t('labs.lab03.replay')}: blocked={not replay_ok}")

    settled = ch.settle()
    trace.log("settle", **settled)
    print(f"{t('ui.result')}: hub={settled['to_hub']} refund={settled['refund']}")

    print(f"\n{t('ui.trace')} ({len(trace)} {t('ui.events')}):")
    for event in trace.events:
        print(" ", event)

    print(f"\n--- {t('exercises.heading')} ---")
    print(t("exercises.lab03_hint"))

main()
